# Створення застосунків для генерації зображень (OpenAI)

Великі мовні моделі (LLM) — це не лише генерація тексту. Ви також можете генерувати зображення за текстовими описами. Зображення як модальність корисні в MedTech, архітектурі, туризмі, розробці ігор, маркетингу та інше. У цьому уроці ми працюємо з сьогоднішніми моделями **GPT Image** на платформі OpenAI.

## Цілі навчання

До кінця цього уроку ви зможете:

- Пояснити, що таке генерація зображень і де вона корисна.
- Зрозуміти сімейство моделей `gpt-image` та чим воно відрізняється від застарілих моделей DALL·E.
- Побудувати застосунок для генерації зображень і редагувати зображення.

## Що таке генерація зображень?

Моделі генерації зображень створюють зображення за текстовим запитом. Сучасні моделі, такі як `gpt-image`, навчаються під час тренування розуміти зв’язок між текстом і зображенням, а потім ітеративно перетворюють випадковий шум у зображення, яке відповідає вашому запиту.

Дві відомі сім’ ї моделей зображень:

- **`gpt-image` (OpenAI)** — поточне покоління, що використовується в цьому уроці. Підтримує генерацію зображень з тексту та редагування зображень (inpainting з маскою).
- **Midjourney** — популярна модель стороннього розробника з власним сервісом і робочим процесом у Discord.

> Старіші моделі зображень OpenAI — **DALL·E 2** і **DALL·E 3** — застарілі. DALL·E 3 більше недоступна для нових розгортань, а функції на кшталт `create_variation` були лише в DALL·E 2. Використовуйте моделі `gpt-image` для нових застосунків.

> **Важливо:** моделі `gpt-image` повертають створене зображення у форматі **base64** (`b64_json`), а не як URL. Ваш код декодує base64-рядок у байти і зберігає — URL для скачування зображення немає.


## Створення вашого першого додатку для генерації зображень

Отже, що потрібно, щоб створити додаток для генерації зображень? Вам потрібні такі бібліотеки:

- **python-dotenv**, дуже рекомендується використовувати цю бібліотеку для зберігання ваших секретів у файлі *.env* окремо від коду.
- **openai**, ця бібліотека використовується для взаємодії з API OpenAI.
- **pillow**, для роботи з зображеннями у Python.


1. Створіть файл *.env* із таким вмістом:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Зберіть наведені вище бібліотеки у файл з назвою *requirements.txt* ось так:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Далі створіть віртуальне оточення та встановіть бібліотеки:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Для Windows використовуйте наступні команди, щоб створити та активувати ваше віртуальне середовище:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Додайте наступний код у файл з назвою *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Створіть об'єкт OpenAI (читає OPENAI_API_KEY з вашого .env)
    client = openai.OpenAI()


    try:
        # Створіть зображення за допомогою API генерації зображень
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Введіть текст підказки тут
            size='1024x1024',
            n=1
        )
        # Встановіть каталог для збереженого зображення
        image_dir = os.path.join(os.curdir, 'images')

        # Якщо каталог не існує, створіть його
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Ініціалізуйте шлях до зображення (зверніть увагу, тип файлу повинен бути png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Моделі gpt-image повертають зображення у форматі base64 (b64_json), а не URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Відобразити зображення у стандартному переглядачі зображень
        image = Image.open(image_path)
        image.show()

    # обробка виключень
    except openai.BadRequestError as err:
        print(err)

    ```

Пояснимо цей код:

- Спочатку ми імпортуємо потрібні бібліотеки, включно з бібліотекою OpenAI, бібліотекою dotenv, модулем base64 та бібліотекою Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Після цього створюємо клієнта, який читає API-ключ із вашого файлу ``.env``.

    ```python
    # Створити об'єкт OpenAI
    client = openai.OpenAI()
    ```

- Далі ми генеруємо зображення:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Введіть текст запиту тут
        size='1024x1024',
        n=1
    )
    ```

    Моделі `gpt-image` повертають зображення у вигляді **base64** рядка в `data[0].b64_json`. Ми декодуємо його в байти та записуємо у файл — URL для завантаження відсутній.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Нарешті, ми відкриваємо зображення і використовуємо стандартний переглядач для його відображення:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Детальніше про генерацію зображення

Розглянемо параметри `images.generate`:

- **model** — модель зображення, яку потрібно використати (наприклад, `gpt-image-1`).
- **prompt** — текстовий запит для генерації зображення. Тут це "Кролик на коні, тримає льодяник, на туманному луці, де ростуть нарциси".
- **size** — розмір генерованого зображення (наприклад, `1024x1024`, `1536x1024`, `1024x1536` або `"auto"`).
- **n** — кількість згенерованих зображень. Тут ми генеруємо одне.

> Моделі для зображень не мають параметра `temperature` — це параметр контролю генерації тексту. Щоб отримати різноманітність, викликайте API знову; щоб зменшити різноманітність, зробіть ваш запит більш конкретним.

## Додаткові можливості генерації зображень

Ви вже бачили, як генерувати зображення кількома рядками Python. Моделі `gpt-image` також можуть **редагувати** існуюче зображення. Надання зображення, необов’язкової **маски** (яка позначає область для зміни) і запиту дозволяють змінити частину зображення — наприклад, додати капелюх нашому кролику.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# редагування також повертаються у форматі base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Початкове зображення містить лише кролика; у фінальному зображенні на кролику капелюх.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
